# 싱글턴 vs 멀티턴, 스트리밍, REST API

Ollama에서 독립 호출과 메시지 히스토리 기반 대화 호출의 차이를 확인합니다.


## 0. 준비


In [1]:
import json
import os

import requests
from dotenv import load_dotenv
from ollama import Client

load_dotenv()

HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL = os.getenv("OLLAMA_MODEL", "gemma3:1b")
client = Client(host=HOST)

print(f"Ollama host: {HOST}")
print(f"사용 모델: {MODEL}")


Ollama host: http://localhost:11434
사용 모델: gemma4:e2b


In [2]:
def response_text(response) -> str:
    message = getattr(response, "message", None)
    if message is not None:
        return getattr(message, "content", "") or ""
    return getattr(response, "response", "") or ""


def print_response(response) -> None:
    text = response_text(response)
    if text:
        print(text)
    else:
        print("답변 본문이 비어 있습니다. think=False 또는 num_predict 값을 확인하세요.")

def safe_create(label: str, messages: list[dict], **kwargs):
    print(f"\n--- {label} ---")
    kwargs.setdefault("think", False)
    try:
        response = client.chat(
            model=MODEL,
            messages=messages,
            **kwargs,
        )
    except Exception as exc:
        print("Ollama 호출 실패:", type(exc).__name__)
        print(exc)
        return None

    print_response(response)
    print(
        f"\n지표: prompt_eval_count={getattr(response, 'prompt_eval_count', None)}, "
        f"eval_count={getattr(response, 'eval_count', None)}, "
        f"total_duration={getattr(response, 'total_duration', None)}"
    )
    return response


## 1. 싱글턴 호출

각 요청이 독립적으로 처리됩니다. 두 번째 요청은 첫 번째 응답을 자동으로 기억하지 않습니다.


In [3]:
single_first = safe_create(
    "싱글턴 1",
    messages=[{"role": "user", "content": "파이썬이 뭐야? 한 문장으로 답해줘."}],
    options={"num_predict": 120},
)



--- 싱글턴 1 ---
파이썬은 사람이 읽기 쉬운 문법을 사용하여 다양한 프로그래밍 작업을 수행할 수 있게 해주는 인기 있는 프로그래밍 언어입니다.

지표: prompt_eval_count=23, eval_count=34, total_duration=3658429900


In [4]:
single_second = safe_create(
    "싱글턴 2",
    messages=[{"role": "user", "content": "방금 말한 것보다 더 쉽게 설명해줘."}],
    options={"num_predict": 120},
)



--- 싱글턴 2 ---
어떤 것을 더 쉽게 설명해 드릴까요? 😊

**제가 방금 했던 이야기나 질문을 다시 말씀해주시면,** 제가 그 내용을 **더 쉽고 간단하게** 바꿔서 설명해 드릴게요!

무엇에 대해 알고 싶으신가요? 편하게 말씀해주세요!

지표: prompt_eval_count=21, eval_count=62, total_duration=4985074000


## 2. 메시지 히스토리로 멀티턴 호출

Ollama는 이전 대화를 `messages` 배열에 직접 넣어 대화를 이어갑니다.


In [5]:
history = [
    {
        "role": "system",
        "content": "당신은 초보자를 가르치는 Python 튜터입니다. 짧고 친절하게 답하세요.",
    }
]


def chat_turn(user_message: str, num_predict: int = 180):
    history.append({"role": "user", "content": user_message})
    response = safe_create(
        f"멀티턴: {user_message}",
        messages=history,
        options={"num_predict": num_predict},
    )
    if response is not None:
        history.append(
            {
                "role": "assistant",
                "content": response_text(response),
            }
        )
    return response


chat_turn("파이썬이 뭐야? 한 문장으로 답해줘.", 120)



--- 멀티턴: 파이썬이 뭐야? 한 문장으로 답해줘. ---
파이썬은 배우기 쉽고 다양한 분야에 사용될 수 있는 인기 있는 프로그래밍 언어예요. 😊

지표: prompt_eval_count=51, eval_count=26, total_duration=3438592000


ChatResponse(model='gemma4:e2b', created_at='2026-07-03T09:36:01.5974109Z', done=True, done_reason='stop', total_duration=3438592000, load_duration=1585084300, prompt_eval_count=51, prompt_eval_duration=460183000, eval_count=26, eval_duration=1372739000, message=Message(role='assistant', content='파이썬은 배우기 쉽고 다양한 분야에 사용될 수 있는 인기 있는 프로그래밍 언어예요. 😊', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [6]:
chat_turn("방금 말한 것보다 더 쉽게 설명해줘.", 160)



--- 멀티턴: 방금 말한 것보다 더 쉽게 설명해줘. ---
파이썬은 컴퓨터에게 우리가 원하는 일을 시키는 방법을 알려주는 **말**과 같아요! 🗣️

지표: prompt_eval_count=98, eval_count=25, total_duration=3438140600


ChatResponse(model='gemma4:e2b', created_at='2026-07-03T09:36:05.0499596Z', done=True, done_reason='stop', total_duration=3438140600, load_duration=1648422000, prompt_eval_count=98, prompt_eval_duration=455513000, eval_count=25, eval_duration=1323925000, message=Message(role='assistant', content='파이썬은 컴퓨터에게 우리가 원하는 일을 시키는 방법을 알려주는 **말**과 같아요! 🗣️', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [7]:
chat_turn("예시 코드도 짧게 줘.", 220)



--- 멀티턴: 예시 코드도 짧게 줘. ---
네, 간단한 "안녕!"을 출력하는 예시를 보여드릴게요.

```python
print("안녕하세요!")
```

이 코드는 화면에 "안녕하세요!"라고 보여줍니다. 😊

지표: prompt_eval_count=141, eval_count=43, total_duration=4633476700


ChatResponse(model='gemma4:e2b', created_at='2026-07-03T09:36:09.6945924Z', done=True, done_reason='stop', total_duration=4633476700, load_duration=1495661400, prompt_eval_count=141, prompt_eval_duration=851498000, eval_count=43, eval_duration=2276647000, message=Message(role='assistant', content='네, 간단한 "안녕!"을 출력하는 예시를 보여드릴게요.\n\n```python\nprint("안녕하세요!")\n```\n\n이 코드는 화면에 "안녕하세요!"라고 보여줍니다. 😊', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [8]:
print("--- 현재 history 대화 기록 ---")
for index, message in enumerate(history, start=1):
    content = message["content"].replace("\n", " ")
    print(f"{index}. {message['role']}: {content[:120]}")


--- 현재 history 대화 기록 ---
1. system: 당신은 초보자를 가르치는 Python 튜터입니다. 짧고 친절하게 답하세요.
2. user: 파이썬이 뭐야? 한 문장으로 답해줘.
3. assistant: 파이썬은 배우기 쉽고 다양한 분야에 사용될 수 있는 인기 있는 프로그래밍 언어예요. 😊
4. user: 방금 말한 것보다 더 쉽게 설명해줘.
5. assistant: 파이썬은 컴퓨터에게 우리가 원하는 일을 시키는 방법을 알려주는 **말**과 같아요! 🗣️
6. user: 예시 코드도 짧게 줘.
7. assistant: 네, 간단한 "안녕!"을 출력하는 예시를 보여드릴게요.  ```python print("안녕하세요!") ```  이 코드는 화면에 "안녕하세요!"라고 보여줍니다. 😊


## 3. 컨텍스트 성장 확인


In [9]:
long_history = [
    {"role": "system", "content": "짧고 명확하게 답하세요."}
]

questions = [
    "로컬 LLM을 안전하게 사용하는 방법을 한 문장으로 알려줘.",
    "방금 답변에서 가장 중요한 규칙 하나만 골라줘.",
    "그 규칙을 어겼을 때 생기는 문제를 예시로 들어줘.",
]

for question in questions:
    long_history.append({"role": "user", "content": question})
    response = safe_create(
        f"컨텍스트 성장: {question}",
        messages=long_history,
        options={"num_predict": 160},
    )
    if response is None:
        break
    long_history.append({"role": "assistant", "content": response_text(response)})
    print(f"현재 메시지 수: {len(long_history)}")



--- 컨텍스트 성장: 로컬 LLM을 안전하게 사용하는 방법을 한 문장으로 알려줘. ---
로컬 LLM 사용 시, 데이터 프라이버시 보호와 악성 코드 방지를 위해 신뢰할 수 있는 출처에서 모델을 다운로드하고 보안 소프트웨어를 사용해야 합니다.

지표: prompt_eval_count=38, eval_count=42, total_duration=3977604700
현재 메시지 수: 3

--- 컨텍스트 성장: 방금 답변에서 가장 중요한 규칙 하나만 골라줘. ---
**신뢰할 수 있는 출처에서 모델을 다운로드하고 보안 소프트웨어를 사용하세요.**

지표: prompt_eval_count=102, eval_count=22, total_duration=3401931900
현재 메시지 수: 5

--- 컨텍스트 성장: 그 규칙을 어겼을 때 생기는 문제를 예시로 들어줘. ---
신뢰할 수 없는 출처에서 모델을 다운로드했을 경우, 해당 모델에 **악성 코드가 포함되어 개인 정보가 유출되거나 시스템이 손상될 수 있습니다.**

지표: prompt_eval_count=149, eval_count=41, total_duration=4734897300
현재 메시지 수: 7


## 4. 스트리밍 응답


In [10]:
def stream_ollama(label: str, messages: list[dict], **kwargs):
    print(f"\n--- {label} ---")
    kwargs.setdefault("think", False)
    printed = False

    try:
        stream = client.chat(
            model=MODEL,
            messages=messages,
            stream=True,
            **kwargs,
        )
        for chunk in stream:
            content = chunk.message.content or ""
            if content:
                printed = True
                print(content, end="", flush=True)

        if printed:
            print()
        else:
            print("답변 본문이 비어 있습니다. think=False 또는 num_predict 값을 확인하세요.")
    except Exception as exc:
        print("Ollama 스트리밍 실패:", type(exc).__name__)
        print(exc)


In [11]:
stream_ollama(
    "스트리밍 단일 응답",
    messages=[
        {"role": "user", "content": "Ollama 스트리밍이 필요한 상황을 3가지로 설명해줘."}
    ],
    options={"num_predict": 260},
)



--- 스트리밍 단일 응답 ---
## Ollama 스트리밍이 필요한 상황 3가지

Ollama를 사용하여 모델 추론(Inference)을 할 때 **스트리밍(Streaming)**은 사용자가 결과를 기다리는 동안 전체 응답이 완료될 때까지 기다리지 않고, **결과가 생성되는 즉시** 사용자에게 데이터를 실시간으로 전달해 주는 방식을 의미합니다. 이는 특히 대규모 언어 모델(LLM)의 응답에서 사용자 경험(UX)을 극대화하고 시스템 효율성을 높이는 데 매우 중요합니다.

다음은 Ollama 스트리밍이 필수적이거나 매우 유용한 세 가지 상황입니다.

---

### 1. 실시간 채팅 및 대화형 인터페이스 (Interactive Chat & Conversational Interfaces)

가장 대표적이고 중요한 사용 사례입니다. 사용자가 질문을 입력했을 때, 모델이 한 단어씩, 혹은 문장 단위로 답변을 생성하는 과정을 사용자에게 즉시 보여줄 때 스트리밍이 필수적입니다.

* **필요성:** 만약 스트리밍 없이 전체 응답이 완료될 때까지 기다린다면, 사용자는 지루함을 느끼고 응답이 느리다고 인식할 수 있습니다.
* **스트리밍의 이점:** 모델


In [12]:
stream_ollama(
    "스트리밍 멀티턴 응답",
    messages=history + [{"role": "user", "content": "지금까지의 설명을 한 문단으로 요약해줘."}],
    options={"num_predict": 180},
)



--- 스트리밍 멀티턴 응답 ---
파이썬은 배우기 쉽고 여러 곳에 쓰이는 프로그래밍 언어로, 컴퓨터에게 원하는 작업을 시키는 방법을 알려주는 도구이며, `print("안녕하세요!")`와 같은 간단한 코드로 시작할 수 있습니다.


## 5. REST API 직접 호출


In [13]:
payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "REST API로 Ollama를 호출하는 장점을 2가지로 설명해줘."}
    ],
    "stream": False,
    "think": False,
    "options": {"num_predict": 180},
}

try:
    response = requests.post(f"{HOST}/api/chat", json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    print(data["message"]["content"])
    print("\n[응답 키]")
    print(sorted(data.keys()))
except requests.RequestException as exc:
    print("REST API 호출 실패:", exc)


## REST API로 Ollama를 호출하는 장점 2가지

REST API를 사용하여 Ollama를 호출하는 것은 다음과 같은 두 가지 주요 이점을 제공합니다.

### 1. **접근성 및 확장성 (Accessibility and Extensibility)**

* **장점 설명:** REST API는 HTTP 프로토콜을 기반으로 하기 때문에, Ollama의 기능을 외부 시스템(웹 애플리케이션, 다른 서비스, 모바일 앱 등)에서 매우 쉽게 접근하고 통합할 수 있게 해줍니다. 이는 **접근성의 극대화**를 의미합니다.
* **구체적인 이점:**
    * **다양한 클라이언트 지원:** Python, JavaScript, Java 등 어떤 프로그래밍 언어를 사용하든 HTTP 요청만 알면 Ollama의 모델 추론(Inference) 기능을 호출할 수 있습니다.
    * **시스템

[응답 키]
['created_at', 'done', 'done_reason', 'eval_count', 'eval_duration', 'load_duration', 'message', 'model', 'prompt_eval_count', 'prompt_eval_duration', 'total_duration']


## 6. REST API 스트리밍


In [14]:
payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "스트리밍 REST 응답을 한 문단으로 설명해줘."}
    ],
    "stream": True,
    "think": False,
    "options": {"num_predict": 180},
}

try:
    with requests.post(f"{HOST}/api/chat", json=payload, stream=True, timeout=120) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line:
                continue
            chunk = json.loads(line)
            print(chunk.get("message", {}).get("content") or "", end="", flush=True)
            if chunk.get("done"):
                print()
except requests.RequestException as exc:
    print("REST 스트리밍 호출 실패:", exc)


스트리밍 REST 응답은 서버가 **전체 데이터를 한 번에 보내지 않고, 데이터를 작은 조각(청크)들로 나누어 요청하는 클라이언트에게 점진적으로 전송**하는 방식입니다. 이를 통해 대용량 파일이나 실시간으로 발생하는 데이터 스트림을 효율적으로 처리할 수 있으며, 클라이언트는 데이터가 도착하는 즉시 처리할 수 있어 **응답 대기 시간을 줄이고 사용자 경험을 향상**시키는 장점이 있습니다.


## 7. 정리

| 방식 | 장점 | 주의점 |
| --- | --- | --- |
| 싱글턴 | 단순하고 빠름 | 이전 대화를 기억하지 않음 |
| 메시지 히스토리 | 대화 맥락 유지 | 매 요청마다 입력 토큰 증가 |
| 스트리밍 | 빠른 체감 응답 | 이벤트 처리 코드 필요 |
| REST API | 언어와 프레임워크 독립적 | JSON 구조와 타임아웃 관리 필요 |
